# Advanced MCMC

In [ ]:
%pip install numpy matplotlib scipy arviz pymc numpyro jax

## Recap and Motivation

In the previous lecture, we developed the foundations of MCMC: Markov chain theory, the Metropolis-Hastings algorithm, Gibbs sampling, and convergence diagnostics. Random walk Metropolis-Hastings (RWMH) is a general-purpose tool, but it has a fundamental limitation: it explores the target distribution by diffusive, undirected random steps. In $d$ dimensions, the step size must shrink as $O(d^{-1/2})$ to maintain a reasonable acceptance rate, and the chain requires $O(d)$ steps to traverse the typical set. The total cost to generate one effectively independent sample scales as $O(d^2)$.

For the Bayesian logistic regression model from the previous lecture ($d = 2$), RWMH works fine. But biostatistical models often have tens, hundreds, or thousands of parameters (hierarchical models, spatial models, survival models with frailties). In these settings, RWMH becomes impractically slow.

This lecture covers two ideas that address this limitation:

1. **Hamiltonian Monte Carlo (HMC):** Uses gradient information to make large, directed proposals that follow the geometry of the target distribution. Cost scales as $O(d^{5/4})$ instead of $O(d^2)$.

2. **Adaptive MCMC:** Automatically learns the scale and correlation structure of the target during the run, eliminating manual tuning.

We then introduce probabilistic programming tools (PyMC, NumPyro) that implement these methods and make Bayesian modeling accessible in practice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

### Running Example: Bayesian Logistic Regression (Continued)

We continue with the Bayesian logistic regression model from the previous lecture:

$$y_i | \boldsymbol{\beta} \sim \text{Bernoulli}(\sigma(\mathbf{x}_i^T \boldsymbol{\beta})), \quad \boldsymbol{\beta} \sim N(\mathbf{0}, \tau^2 \mathbf{I})$$

In [ ]:
# Simulate data (same as previous lecture)
n = 200
p = 2
X = np.column_stack([np.ones(n), np.random.normal(0, 1, n)])
beta_true = np.array([0.5, 1.5])
prob = 1 / (1 + np.exp(-X @ beta_true))
y = np.random.binomial(1, prob)

def log_posterior(beta, X, y, tau=10.0):
    """Log-posterior for Bayesian logistic regression (up to a constant)."""
    eta = X @ beta
    log_lik = np.sum(y * eta - np.logaddexp(0, eta))
    log_prior = -0.5 * np.sum(beta**2) / tau**2
    return log_lik + log_prior

def grad_log_posterior(beta, X, y, tau=10.0):
    """Gradient of the log-posterior."""
    eta = X @ beta
    p_hat = 1 / (1 + np.exp(-eta))
    grad_lik = X.T @ (y - p_hat)
    grad_prior = -beta / tau**2
    return grad_lik + grad_prior

print(f"True coefficients: {beta_true}")

## Hamiltonian Monte Carlo

### Intuition: Physics to the Rescue

Imagine placing a frictionless puck on a curved surface where the height at position $\boldsymbol{\theta}$ equals $-\log \pi(\boldsymbol{\theta})$. The valleys of this surface correspond to high-density regions of the posterior. If we give the puck a random kick, it will roll across the surface, speed up in the valleys (high posterior density) and slow down on the hills (low density). After some time, we record its position as a new MCMC sample.

Because total energy (kinetic + potential) is approximately conserved, the puck ends up in a region of similar posterior density but potentially far from where it started. This is the key insight of HMC: by simulating Hamiltonian dynamics, we can propose moves that are both large and likely to be accepted.

### The Hamiltonian Framework

To formalize this, we introduce auxiliary **momentum** variables $\mathbf{r} \in \mathbb{R}^d$ (one per parameter) and define the **Hamiltonian**:

$$H(\boldsymbol{\theta}, \mathbf{r}) = U(\boldsymbol{\theta}) + K(\mathbf{r})$$

where:

- $U(\boldsymbol{\theta}) = -\log \pi(\boldsymbol{\theta})$ is the **potential energy** (negative log-posterior)
- $K(\mathbf{r}) = \frac{1}{2} \mathbf{r}^T \mathbf{M}^{-1} \mathbf{r}$ is the **kinetic energy**, with **mass matrix** $\mathbf{M}$

The joint distribution over $(\boldsymbol{\theta}, \mathbf{r})$ is:

$$p(\boldsymbol{\theta}, \mathbf{r}) \propto \exp(-H(\boldsymbol{\theta}, \mathbf{r})) = \underbrace{\pi(\boldsymbol{\theta})}_{\text{target}} \cdot \underbrace{\exp(-\tfrac{1}{2} \mathbf{r}^T \mathbf{M}^{-1} \mathbf{r})}_{\text{Gaussian in } \mathbf{r}}$$

Because the joint factorizes, the marginal distribution of $\boldsymbol{\theta}$ is exactly the target $\pi(\boldsymbol{\theta})$. The momentum $\mathbf{r}$ is simply a computational device.

**Hamilton's equations** describe how $\boldsymbol{\theta}$ and $\mathbf{r}$ evolve over continuous time:

$$\frac{d\boldsymbol{\theta}}{dt} = \mathbf{M}^{-1}\mathbf{r}, \qquad \frac{d\mathbf{r}}{dt} = -\nabla U(\boldsymbol{\theta}) = \nabla \log \pi(\boldsymbol{\theta})$$

These dynamics have three properties that make them ideal for MCMC:

1. **Energy conservation:** $H(\boldsymbol{\theta}(t), \mathbf{r}(t))$ is constant along trajectories. A proposal from a Hamiltonian trajectory would always be accepted.
2. **Time reversibility:** Negating $\mathbf{r}$ and running backward retraces the trajectory. This guarantees detailed balance.
3. **Volume preservation:** Phase-space volumes are preserved (Liouville's theorem), so no Jacobian correction is needed in the acceptance ratio.

### Question

In the HMC framework, why do we introduce momentum variables $\mathbf{r}$? What would happen if we tried to use only the gradient $\nabla \log \pi(\boldsymbol{\theta})$ to make proposals without the momentum?

### Answer

The momentum variables serve two purposes. First, they provide inertia that carries the proposal trajectory through regions of low gradient, allowing the chain to cross "flat" parts of the posterior and make large moves. Without momentum, a gradient-only method would simply climb to the nearest mode and stay there (this is gradient ascent, not sampling). Second, the momentum creates a joint distribution $p(\boldsymbol{\theta}, \mathbf{r})$ whose marginal in $\boldsymbol{\theta}$ is exactly the target. By refreshing $\mathbf{r}$ randomly at each iteration and simulating the joint dynamics, we obtain samples from the correct target distribution. A gradient-only approach (like the Langevin algorithm without the noise term) would converge to a point, not a distribution.

### The Leapfrog Integrator

We cannot solve Hamilton's equations exactly for general posteriors, so we use a numerical integrator. The **leapfrog** (Stormer-Verlet) integrator is the standard choice. One leapfrog step with step size $\varepsilon$:

$$\mathbf{r}_{1/2} = \mathbf{r}_0 + \frac{\varepsilon}{2} \nabla \log \pi(\boldsymbol{\theta}_0) \qquad \text{(half-step in momentum)}$$

$$\boldsymbol{\theta}_1 = \boldsymbol{\theta}_0 + \varepsilon \, \mathbf{M}^{-1} \mathbf{r}_{1/2} \qquad \text{(full step in position)}$$

$$\mathbf{r}_1 = \mathbf{r}_{1/2} + \frac{\varepsilon}{2} \nabla \log \pi(\boldsymbol{\theta}_1) \qquad \text{(half-step in momentum)}$$

For $L$ consecutive leapfrog steps, the intermediate half-steps combine, so we only compute one gradient per step (not two):

```
Full leapfrog trajectory (L steps):
  r ← r + (ε/2) ∇log π(θ)
  For l = 1, ..., L-1:
    θ ← θ + ε M⁻¹ r
    r ← r + ε ∇log π(θ)
  θ ← θ + ε M⁻¹ r
  r ← r + (ε/2) ∇log π(θ)
```

Why leapfrog instead of simpler methods like Euler's method? The leapfrog integrator is **symplectic**: it exactly preserves phase-space volume even though it only approximately conserves energy. The energy error is bounded and oscillatory rather than drifting monotonically, which is critical for maintaining high acceptance rates. Leapfrog has $O(\varepsilon^2)$ global error over a fixed trajectory length, compared to $O(\varepsilon)$ for Euler.

In [ ]:
def leapfrog(theta, r, grad_log_pi, epsilon, L, M_inv=None):
    """Leapfrog integrator for Hamiltonian dynamics.

    Parameters
    ----------
    theta : array of shape (d,)
        Current position.
    r : array of shape (d,)
        Current momentum.
    grad_log_pi : callable
        Gradient of log target density.
    epsilon : float
        Step size.
    L : int
        Number of leapfrog steps.
    M_inv : array of shape (d,), optional
        Diagonal of inverse mass matrix. Default: identity.

    Returns
    -------
    theta_new, r_new : arrays of shape (d,)
    """
    theta = theta.copy()
    r = r.copy()

    if M_inv is None:
        M_inv = np.ones(len(theta))  # Identity (diagonal)

    # Half step for momentum
    r = r + 0.5 * epsilon * grad_log_pi(theta)

    for l in range(L - 1):
        # Full step for position
        theta = theta + epsilon * M_inv * r
        # Full step for momentum
        r = r + epsilon * grad_log_pi(theta)

    # Full step for position
    theta = theta + epsilon * M_inv * r
    # Half step for momentum
    r = r + 0.5 * epsilon * grad_log_pi(theta)

    return theta, r

### Visualizing Leapfrog Trajectories

To build intuition, let us trace a leapfrog trajectory on a 2D target distribution and compare it to random walk proposals:

In [ ]:
# Trace a leapfrog trajectory on the logistic regression posterior
theta_start = np.array([0.3, 1.0])
rng_hmc = np.random.default_rng(42)
r_start = rng_hmc.normal(0, 1, size=p)

# Collect intermediate positions
epsilon_demo = 0.05
L_demo = 40
trajectory = [theta_start.copy()]
theta_traj, r_traj = theta_start.copy(), r_start.copy()

r_traj = r_traj + 0.5 * epsilon_demo * grad_log_posterior(theta_traj, X, y)
for l in range(L_demo):
    theta_traj = theta_traj + epsilon_demo * r_traj
    if l < L_demo - 1:
        r_traj = r_traj + epsilon_demo * grad_log_posterior(theta_traj, X, y)
    else:
        r_traj = r_traj + 0.5 * epsilon_demo * grad_log_posterior(theta_traj, X, y)
    trajectory.append(theta_traj.copy())

trajectory = np.array(trajectory)

# Compare: random walk proposals from the same starting point
rw_proposals = theta_start + rng_hmc.normal(0, 0.15, size=(20, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Posterior contour
b0_grid = np.linspace(-0.2, 1.2, 100)
b1_grid = np.linspace(0.5, 2.5, 100)
B0, B1 = np.meshgrid(b0_grid, b1_grid)
log_post_grid = np.array([
    log_posterior(np.array([b0, b1]), X, y)
    for b0, b1 in zip(B0.ravel(), B1.ravel())
]).reshape(B0.shape)

for ax in axes:
    ax.contour(B0, B1, np.exp(log_post_grid - log_post_grid.max()),
               levels=10, colors="gray", alpha=0.5)
    ax.set_xlabel("$\\beta_0$")
    ax.set_ylabel("$\\beta_1$")

# HMC trajectory
axes[0].plot(trajectory[:, 0], trajectory[:, 1], "o-", markersize=3,
             linewidth=1, color="steelblue", alpha=0.8)
axes[0].plot(*trajectory[0], "rs", markersize=10, label="Start")
axes[0].plot(*trajectory[-1], "g^", markersize=10, label="Proposal")
axes[0].set_title("HMC: Leapfrog Trajectory")
axes[0].legend()

# Random walk proposals
axes[1].plot(*theta_start, "rs", markersize=10, label="Current", zorder=5)
for i in range(len(rw_proposals)):
    axes[1].annotate("", xy=rw_proposals[i], xytext=theta_start,
                     arrowprops=dict(arrowstyle="->", color="tab:orange",
                                     alpha=0.5, lw=0.8))
axes[1].scatter(rw_proposals[:, 0], rw_proposals[:, 1], s=20,
                color="tab:orange", alpha=0.7, label="Proposals")
axes[1].set_title("Random Walk MH: Proposals")
axes[1].legend()

plt.tight_layout()

The HMC trajectory follows the contours of the posterior, ending far from the start but in a region of similar density. The random walk proposals scatter in all directions, with many landing in low-density regions that will be rejected.

### The HMC Algorithm

Putting the pieces together, one iteration of HMC:

1. **Resample momentum:** Draw $\mathbf{r} \sim N(\mathbf{0}, \mathbf{M})$.
2. **Simulate dynamics:** Run $L$ leapfrog steps with step size $\varepsilon$ to get proposal $(\boldsymbol{\theta}^*, \mathbf{r}^*)$.
3. **Metropolis correction:** Accept $\boldsymbol{\theta}^*$ with probability $\min(1, \exp(-H(\boldsymbol{\theta}^*, \mathbf{r}^*) + H(\boldsymbol{\theta}, \mathbf{r})))$.

If the leapfrog integrator were exact (zero energy error), the acceptance probability would always be 1. The Metropolis correction accounts for the numerical integration error and ensures the chain targets the exact posterior.

In [ ]:
def hmc(log_pi, grad_log_pi, x0, epsilon, L, n_iter, M_inv=None, rng=None):
    """Hamiltonian Monte Carlo sampler.

    Parameters
    ----------
    log_pi : callable
        Log target density (up to a constant).
    grad_log_pi : callable
        Gradient of log target density.
    x0 : array of shape (d,)
        Initial state.
    epsilon : float
        Leapfrog step size.
    L : int
        Number of leapfrog steps per iteration.
    n_iter : int
        Number of HMC iterations.
    M_inv : array of shape (d,), optional
        Diagonal inverse mass matrix.
    rng : numpy random Generator, optional

    Returns
    -------
    dict with keys: samples, acceptance_rate, log_target_values
    """
    if rng is None:
        rng = np.random.default_rng()
    d = len(x0)
    if M_inv is None:
        M_inv = np.ones(d)
    M = 1.0 / M_inv  # Diagonal mass matrix

    samples = np.zeros((n_iter, d))
    log_target_values = np.zeros(n_iter)
    n_accept = 0

    theta = x0.copy()
    log_pi_current = log_pi(theta)

    for t in range(n_iter):
        # 1. Resample momentum
        r = rng.normal(0, np.sqrt(M))

        # 2. Leapfrog integration
        theta_prop, r_prop = leapfrog(theta, r, grad_log_pi, epsilon, L, M_inv)

        # 3. Compute Hamiltonian (potential + kinetic energy)
        log_pi_prop = log_pi(theta_prop)
        kinetic_current = 0.5 * np.sum(r**2 * M_inv)
        kinetic_prop = 0.5 * np.sum(r_prop**2 * M_inv)
        H_current = -log_pi_current + kinetic_current
        H_prop = -log_pi_prop + kinetic_prop

        # 4. Metropolis accept/reject
        if np.log(rng.uniform()) < H_current - H_prop:
            theta = theta_prop
            log_pi_current = log_pi_prop
            n_accept += 1

        samples[t] = theta
        log_target_values[t] = log_pi_current

    return {
        "samples": samples,
        "acceptance_rate": n_accept / n_iter,
        "log_target_values": log_target_values,
    }

### HMC on Bayesian Logistic Regression

In [ ]:
rng_hmc_run = np.random.default_rng(42)
result_hmc = hmc(
    log_pi=lambda b: log_posterior(b, X, y),
    grad_log_pi=lambda b: grad_log_posterior(b, X, y),
    x0=np.zeros(p),
    epsilon=0.05,
    L=20,
    n_iter=5000,
    rng=rng_hmc_run,
)

print(f"HMC acceptance rate: {result_hmc['acceptance_rate']:.3f}")
print(f"Posterior mean: {result_hmc['samples'][500:].mean(axis=0)}")
print(f"Posterior std:  {result_hmc['samples'][500:].std(axis=0)}")
print(f"True beta:      {beta_true}")

### Comparing HMC to Random Walk MH

In [ ]:
# Run RWMH for comparison
def metropolis_hastings(log_target, x0, proposal_sd, n_iter, rng=None):
    """Random walk Metropolis-Hastings sampler."""
    if rng is None:
        rng = np.random.default_rng()
    d = len(x0)
    samples = np.zeros((n_iter, d))
    n_accept = 0
    x_current = x0.copy()
    log_pi_current = log_target(x_current)

    for t in range(n_iter):
        x_proposed = x_current + rng.normal(0, proposal_sd, size=d)
        log_pi_proposed = log_target(x_proposed)
        if np.log(rng.uniform()) < log_pi_proposed - log_pi_current:
            x_current = x_proposed
            log_pi_current = log_pi_proposed
            n_accept += 1
        samples[t] = x_current

    return {"samples": samples, "acceptance_rate": n_accept / n_iter}


def compute_ess(chain):
    """Compute effective sample size using Geyer's initial positive sequence."""
    n_samp = len(chain)
    chain_centered = chain - chain.mean()
    fft_result = np.fft.fft(chain_centered, n=2 * n_samp)
    acf_full = np.real(np.fft.ifft(fft_result * np.conj(fft_result)))[:n_samp]
    acf_full = acf_full / acf_full[0]
    tau = -1.0
    for k in range(0, n_samp - 1, 2):
        pair_sum = acf_full[k] + (acf_full[k + 1] if k + 1 < n_samp else 0.0)
        if pair_sum < 0:
            break
        tau += 2 * pair_sum
    return n_samp / max(tau, 1.0)


rng_mh = np.random.default_rng(42)
result_mh = metropolis_hastings(
    lambda b: log_posterior(b, X, y), np.zeros(p), 0.15, 5000, rng_mh
)

burnin = 500
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Trace plots
for j, (ax_row, label) in enumerate(zip([axes[0], axes[1]],
                                         ["$\\beta_0$", "$\\beta_1$"])):
    ax_row[0].plot(result_mh["samples"][:, j], linewidth=0.4,
                   color="tab:orange", alpha=0.7)
    ax_row[0].axhline(beta_true[j], color="red", linestyle="--")
    ax_row[0].set_title(f"RWMH: {label}")
    ax_row[0].set_ylabel(label)

    ax_row[1].plot(result_hmc["samples"][:, j], linewidth=0.4,
                   color="steelblue", alpha=0.7)
    ax_row[1].axhline(beta_true[j], color="red", linestyle="--")
    ax_row[1].set_title(f"HMC: {label}")

axes[1, 0].set_xlabel("Iteration")
axes[1, 1].set_xlabel("Iteration")
plt.tight_layout()

In [ ]:
# ESS comparison
print("Effective Sample Size (post burn-in, 4500 samples):")
for j in range(p):
    ess_mh = compute_ess(result_mh["samples"][burnin:, j])
    ess_hmc = compute_ess(result_hmc["samples"][burnin:, j])
    print(f"  beta_{j}: RWMH ESS = {ess_mh:.0f}, HMC ESS = {ess_hmc:.0f}")

HMC typically achieves ESS close to the total number of post burn-in samples (low autocorrelation), while RWMH has much lower ESS due to correlated samples. Each HMC iteration is more expensive (requiring $L$ gradient evaluations), but the per-gradient efficiency is still much higher.

### Question

Consider running HMC with step size $\varepsilon = 0.5$ and $L = 100$ leapfrog steps on a posterior that has a narrow funnel-shaped region.

(a) What would you expect to happen to the acceptance rate and why?

(b) If instead you set $\varepsilon = 0.001$ and $L = 1$, what does HMC reduce to?

### Answer

(a) With a large step size of 0.5 in a region with high curvature (like a funnel), the leapfrog integrator accumulates large energy errors. The numerical trajectory diverges from the true Hamiltonian trajectory, and the proposed state has much higher energy than the starting state. The acceptance probability $\exp(H_{\text{current}} - H_{\text{proposal}})$ becomes very small, so most proposals are rejected. This is called a **divergent transition** in practice.

(b) With $L = 1$ leapfrog step, HMC makes a single gradient-informed position update followed by a Metropolis correction. This is closely related to the **Metropolis-adjusted Langevin algorithm (MALA)**, which uses a gradient-informed proposal $\boldsymbol{\theta}^* = \boldsymbol{\theta}_t + \frac{\varepsilon^2}{2} \nabla \log \pi(\boldsymbol{\theta}_t) + \varepsilon \boldsymbol{z}$ where $\boldsymbol{z} \sim N(\mathbf{0}, \mathbf{I})$. The correspondence is not exact (the leapfrog step has a slightly different form than the MALA proposal), but the behavior is qualitatively the same: a single gradient-guided step with a Metropolis correction. With a single leapfrog step, HMC does not build up the long, directed trajectories that give it its advantage over random walks.

### Tuning HMC: Step Size and Trajectory Length

HMC has two tuning parameters: the step size $\varepsilon$ and the number of leapfrog steps $L$.

**Step size $\varepsilon$:** Too large causes divergent trajectories with low acceptance. Too small produces small steps and high autocorrelation. The optimal acceptance rate for HMC is approximately **65%** (compared to 23% for random walk MH), derived from asymptotic scaling theory.

**Number of steps $L$:** Too small limits exploration. Too large wastes computation: the trajectory "doubles back" on itself (a U-turn) and the proposal ends up close to the start. The product $\varepsilon \cdot L$ determines the trajectory length in parameter space and should be roughly comparable to the distance the chain needs to travel across the posterior.

In [ ]:
# Demonstrate the effect of L (trajectory length)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, L_val in zip(axes, [1, 20, 200]):
    rng_L = np.random.default_rng(42)
    res_L = hmc(
        lambda b: log_posterior(b, X, y),
        lambda b: grad_log_posterior(b, X, y),
        np.zeros(p), epsilon=0.05, L=L_val, n_iter=2000, rng=rng_L,
    )
    ax.plot(res_L["samples"][:, 1], linewidth=0.5, color="steelblue")
    ax.axhline(beta_true[1], color="red", linestyle="--")
    ess_val = compute_ess(res_L["samples"][200:, 1])
    ax.set_title(f"L={L_val}, accept={res_L['acceptance_rate']:.0%}, "
                 f"ESS={ess_val:.0f}")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("$\\beta_1$")

plt.tight_layout()

With $L = 1$, the chain behaves similarly to MALA and mixes slowly. With $L = 20$, the trajectory is long enough to cross the posterior and mixing is efficient. With $L = 200$, the chain wastes computation on overly long trajectories that double back, though mixing is still reasonable because the Metropolis correction prevents bad proposals from being accepted.

## The No-U-Turn Sampler (NUTS)

Choosing a good value of $L$ for HMC is problem-specific and tedious. The **No-U-Turn Sampler** (NUTS), introduced by Hoffman and Gelman (2014), automates this by extending the trajectory until it starts to turn back toward the starting point.

### The U-Turn Criterion

NUTS detects when to stop by checking:

$$(\boldsymbol{\theta}(t) - \boldsymbol{\theta}_0) \cdot \mathbf{r}(t) < 0$$

When the momentum $\mathbf{r}(t)$ points back toward the starting position $\boldsymbol{\theta}_0$, the trajectory is curving around, and further integration would bring it closer to the start rather than exploring new territory.

### Tree-Doubling

NUTS builds the trajectory using a **tree-doubling** procedure. Starting from $(\boldsymbol{\theta}_0, \mathbf{r}_0)$:

1. At depth $j = 0$, take one leapfrog step (forward or backward, chosen randomly).
2. At depth $j$, double the trajectory by taking $2^{j-1}$ additional leapfrog steps in the chosen direction.
3. At each doubling, check the U-turn criterion at the endpoints of the subtree and the overall tree.
4. Stop when a U-turn is detected or the tree reaches a maximum depth.
5. Select the proposal from the trajectory using multinomial weighting by $\exp(-H)$.

The bidirectional construction (randomly extending forward or backward) is needed to maintain detailed balance. Each doubling doubles the trajectory length, so after $j$ doublings the trajectory contains $2^j$ leapfrog steps. A maximum tree depth of 10 (1024 steps) is the default in most software.

We will not implement NUTS from scratch because the tree-doubling logic is complex. Instead, we will use it through probabilistic programming libraries. The key takeaway is that NUTS removes the need to tune $L$ and is the default sampler in Stan, PyMC, and NumPyro.

### Question

NUTS uses the criterion $(\boldsymbol{\theta}(t) - \boldsymbol{\theta}_0) \cdot \mathbf{r}(t) < 0$ to detect U-turns.

(a) For a 1D standard normal target $\pi(\theta) = N(0, 1)$, the Hamiltonian trajectory traces an ellipse in $(\theta, r)$ space. After what fraction of the orbit does the U-turn criterion trigger?

(b) Why is it important that NUTS extends the trajectory in *both* directions (forward and backward) rather than only forward?

### Answer

(a) For a 1D standard normal, the Hamiltonian is $H = \theta^2/2 + r^2/2$, and the dynamics trace a circle: $\theta(t) = \theta_0 \cos t + r_0 \sin t$, $r(t) = -\theta_0 \sin t + r_0 \cos t$. The U-turn criterion $(\theta(t) - \theta_0) \cdot r(t) < 0$ triggers when the displacement from the start and the current momentum point in opposite directions, meaning the trajectory is heading back. The exact time at which this happens depends on the initial conditions $(\theta_0, r_0)$, so there is no single fixed fraction of the orbit. The key insight is that the criterion stops the trajectory once it has started to return, preventing wasted computation from completing the full loop back to the start.

(b) If NUTS only extended forward, the selection of the proposal would be biased toward states later in the trajectory. This violates detailed balance because the reverse transition (from the proposal back to the start) would not have the same probability. The bidirectional construction ensures that the trajectory is symmetric in a sense that preserves the correct stationary distribution. Without this, the sampler would produce biased samples.

## Adaptive MCMC

### Motivation

Recall from the previous lecture that the proposal standard deviation in random walk MH must be carefully tuned: too small gives slow exploration, too large gives low acceptance. In higher dimensions, tuning becomes harder because each parameter may have a different scale and parameters may be correlated. **Adaptive MCMC** solves this by learning the proposal distribution from the chain's own history.

### The Adaptive Metropolis Algorithm

Haario, Saksman, and Tamminen (2001) proposed a simple but powerful idea: use the empirical covariance of the samples seen so far as the proposal covariance. After an initial period of $n_0$ iterations (using a fixed proposal), at iteration $t > n_0$:

$$\text{Propose: } \boldsymbol{\theta}^* \sim N\!\left(\boldsymbol{\theta}_t,\; \frac{2.38^2}{d} \hat{\boldsymbol{\Sigma}}_t + \frac{2.38^2}{d} \varepsilon \mathbf{I}_d\right)$$

where:

- $\hat{\boldsymbol{\Sigma}}_t = \text{Cov}(\boldsymbol{\theta}_1, \ldots, \boldsymbol{\theta}_t)$ is the sample covariance of all samples so far
- The scaling factor $2.38^2/d$ comes from the optimal scaling theory for random walk MH (Roberts et al., 1997)
- $\varepsilon \mathbf{I}_d$ is a small regularization term ensuring positive definiteness

The sample covariance can be updated recursively using Welford's online algorithm, avoiding the need to store all past samples.

### Why Does Adaptation Not Break MCMC?

Adapting the proposal based on past samples makes the chain non-Markovian: the transition at step $t$ depends on the entire history $(\boldsymbol{\theta}_1, \ldots, \boldsymbol{\theta}_{t-1})$. This seems like it could break the theoretical guarantees. Roberts and Rosenthal (2007) identified two conditions that ensure the adapted chain still converges to the correct target:

1. **Diminishing adaptation:** The amount of change in the transition kernel at step $n$ converges to zero. In the Haario algorithm, this holds automatically because each new sample's influence on $\hat{\boldsymbol{\Sigma}}_t$ shrinks as $O(1/t)$.

2. **Containment:** The mixing times of the sequence of adapted kernels are bounded in probability. This is a technical condition that holds for most well-behaved models.

In [ ]:
def adaptive_metropolis(log_target, x0, n_iter, n_adapt_start=100,
                        initial_sd=0.1, eps=1e-6, rng=None):
    """Adaptive Metropolis sampler (Haario et al., 2001).

    Parameters
    ----------
    log_target : callable
        Log target density.
    x0 : array of shape (d,)
        Initial state.
    n_iter : int
        Number of iterations.
    n_adapt_start : int
        Start adapting after this many iterations.
    initial_sd : float
        Initial proposal standard deviation (before adaptation).
    eps : float
        Regularization for proposal covariance.
    rng : numpy random Generator, optional

    Returns
    -------
    dict with keys: samples, acceptance_rate
    """
    if rng is None:
        rng = np.random.default_rng()
    d = len(x0)
    sd = 2.38**2 / d  # Optimal scaling factor

    samples = np.zeros((n_iter, d))
    n_accept = 0

    x_current = x0.copy()
    log_pi_current = log_target(x_current)

    # Running statistics for online covariance (Welford's algorithm)
    mean = x0.copy()
    M2 = np.zeros((d, d))

    for t in range(n_iter):
        if t < n_adapt_start:
            # Fixed isotropic proposal before adaptation
            proposal_cov = initial_sd**2 * np.eye(d)
        else:
            # Adaptive proposal: scaled empirical covariance + regularization
            cov_est = M2 / max(t - 1, 1)
            proposal_cov = sd * (cov_est + eps * np.eye(d))

        # Propose
        x_proposed = rng.multivariate_normal(x_current, proposal_cov)

        # Accept/reject
        log_pi_proposed = log_target(x_proposed)
        if np.log(rng.uniform()) < log_pi_proposed - log_pi_current:
            x_current = x_proposed
            log_pi_current = log_pi_proposed
            n_accept += 1

        samples[t] = x_current

        # Update running mean and covariance (Welford's)
        delta = x_current - mean
        mean = mean + delta / (t + 1)
        delta2 = x_current - mean
        M2 = M2 + np.outer(delta, delta2)

    return {"samples": samples, "acceptance_rate": n_accept / n_iter}

### Comparing Fixed vs. Adaptive Proposals

In [ ]:
# Fixed proposal (well-tuned)
rng_fixed = np.random.default_rng(42)
res_fixed = metropolis_hastings(
    lambda b: log_posterior(b, X, y), np.zeros(p), 0.15, 10000, rng_fixed
)

# Fixed proposal (poorly tuned)
rng_bad = np.random.default_rng(42)
res_bad = metropolis_hastings(
    lambda b: log_posterior(b, X, y), np.zeros(p), 2.0, 10000, rng_bad
)

# Adaptive proposal (starting with bad initial scale)
rng_adapt = np.random.default_rng(42)
res_adapt = adaptive_metropolis(
    lambda b: log_posterior(b, X, y), np.zeros(p), 10000,
    initial_sd=2.0, rng=rng_adapt
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ["Fixed (well-tuned, $\\sigma$=0.15)",
          "Fixed (poorly tuned, $\\sigma$=2.0)",
          "Adaptive (starts $\\sigma$=2.0)"]
results = [res_fixed, res_bad, res_adapt]
colors = ["steelblue", "tab:orange", "tab:green"]

for ax, title, res, color in zip(axes, titles, results, colors):
    ax.plot(res["samples"][:, 1], linewidth=0.4, color=color, alpha=0.7)
    ax.axhline(beta_true[1], color="red", linestyle="--")
    ess = compute_ess(res["samples"][2000:, 1])
    ax.set_title(f"{title}\naccept={res['acceptance_rate']:.0%}, ESS={ess:.0f}")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("$\\beta_1$")

plt.tight_layout()

The adaptive sampler recovers from a poor initial proposal scale by learning the posterior covariance. After the adaptation period, its mixing is comparable to the well-tuned fixed proposal.

### Simple Step-Size Adaptation via Robbins-Monro

A simpler form of adaptation targets only the scalar proposal scale. The **Robbins-Monro** stochastic approximation rule updates the log-scale based on whether the current acceptance rate exceeds the target:

$$\log \sigma_{t+1} = \log \sigma_t + a_t \left(\alpha_t - \alpha^*\right)$$

where $\alpha_t$ is the acceptance indicator at step $t$, $\alpha^*$ is the target acceptance rate (e.g., 0.234), and $\{a_t\}$ is a step-size sequence satisfying $\sum a_t = \infty$ and $\sum a_t^2 < \infty$ (e.g., $a_t = 1/t$). The first condition ensures adaptation can reach the optimum, and the second ensures the updates eventually stabilize.

In [ ]:
def rm_adaptive_mh(log_target, x0, n_iter, target_accept=0.234, rng=None):
    """Random walk MH with Robbins-Monro step-size adaptation."""
    if rng is None:
        rng = np.random.default_rng()
    d = len(x0)
    samples = np.zeros((n_iter, d))
    n_accept = 0

    x_current = x0.copy()
    log_pi_current = log_target(x_current)
    log_sigma = np.log(0.1)  # Initial scale

    for t in range(n_iter):
        sigma = np.exp(log_sigma)
        x_proposed = x_current + rng.normal(0, sigma, size=d)
        log_pi_proposed = log_target(x_proposed)
        accept = np.log(rng.uniform()) < log_pi_proposed - log_pi_current

        if accept:
            x_current = x_proposed
            log_pi_current = log_pi_proposed
            n_accept += 1

        samples[t] = x_current

        # Robbins-Monro update
        step = 1.0 / (t + 1)
        log_sigma += step * (float(accept) - target_accept)

    return {
        "samples": samples,
        "acceptance_rate": n_accept / n_iter,
        "final_sigma": np.exp(log_sigma),
    }


rng_rm = np.random.default_rng(42)
res_rm = rm_adaptive_mh(lambda b: log_posterior(b, X, y), np.zeros(p), 10000,
                        rng=rng_rm)
print(f"Robbins-Monro: acceptance={res_rm['acceptance_rate']:.2%}, "
      f"final sigma={res_rm['final_sigma']:.4f}")

### Question

The Adaptive Metropolis algorithm uses the scaling factor $c_d = 2.38^2 / d$.

(a) Where does this number come from? What happens to the proposal covariance as $d$ increases?

(b) Why is the regularization term $\varepsilon \mathbf{I}_d$ needed? What could go wrong without it?

### Answer

(a) The factor $2.38^2/d$ comes from the optimal scaling result of Roberts, Gelman, and Gilks (1997), who showed that for a $d$-dimensional Gaussian target, the optimal random walk MH proposal has covariance $(2.38^2/d) \boldsymbol{\Sigma}$, where $\boldsymbol{\Sigma}$ is the target covariance. This achieves the optimal acceptance rate of approximately 23.4%. As $d$ increases, the proposal must shrink to maintain a reasonable acceptance rate, consistent with the $O(d^{-1/2})$ scaling of the step size.

(b) The regularization $\varepsilon \mathbf{I}_d$ ensures the proposal covariance is positive definite. In the early phase of adaptation, the empirical covariance $\hat{\boldsymbol{\Sigma}}_t$ may be singular or nearly singular: if $t < d$, the sample covariance is rank-deficient (fewer samples than dimensions), and even for $t > d$, the covariance could be ill-conditioned if the samples are concentrated in a low-dimensional subspace. A singular covariance would make it impossible to sample from the multivariate normal proposal.

## Practical MCMC Software

Writing MCMC samplers from scratch, as we have done in this and the previous lecture, is valuable for understanding. But for applied work, probabilistic programming languages provide tested, optimized implementations with automatic tuning. These tools implement NUTS (which handles step size and trajectory length automatically) and mass matrix adaptation, relieving the user of manual tuning.

### PyMC

PyMC is a Python library for Bayesian modeling that uses a context-manager syntax for model specification. It defaults to NUTS for continuous parameters.

In [ ]:
import pymc as pm
import arviz as az

# Bayesian logistic regression in PyMC
with pm.Model() as logistic_model:
    # Priors
    beta = pm.Normal("beta", mu=0, sigma=10, shape=p)

    # Linear predictor
    eta = pm.math.dot(X, beta)

    # Likelihood
    pm.Bernoulli("y_obs", logit_p=eta, observed=y)

    # Sample using NUTS
    trace = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                      progressbar=False)

The `tune` parameter sets the number of warmup iterations during which the step size and mass matrix are adapted. The `draws` parameter (first argument) sets the number of post-warmup samples per chain. After sampling, we use ArviZ for diagnostics:

In [ ]:
# Posterior summary with diagnostics
summary = az.summary(trace, var_names=["beta"])
print(summary)
print(f"\nTrue beta: {beta_true}")

The summary table includes the posterior mean, standard deviation, highest density interval (HDI), ESS, and $\hat{R}$ for each parameter.

In [ ]:
# Trace plot and posterior distributions
az.plot_trace(trace, var_names=["beta"])
plt.tight_layout()

In [ ]:
# Forest plot comparing posterior to true values
az.plot_forest(trace, var_names=["beta"], combined=True)
plt.axvline(beta_true[0], color="red", linestyle="--", alpha=0.5)
plt.axvline(beta_true[1], color="red", linestyle="--", alpha=0.5)
plt.tight_layout()

### Hierarchical Model in PyMC

The real power of probabilistic programming is building complex models declaratively. Here is the Poisson-lognormal GLMM from the previous lecture, which required a custom Metropolis-within-Gibbs sampler:

In [ ]:
# Poisson-lognormal GLMM (same data as previous lecture)
np.random.seed(42)
n_pois = 50
beta0_true_pois = 1.0
sigma_true_pois = 0.8
b_true_pois = np.random.normal(0, sigma_true_pois, n_pois)
y_pois = np.random.poisson(np.exp(beta0_true_pois + b_true_pois))

with pm.Model() as poisson_model:
    # Priors
    beta0 = pm.Normal("beta0", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=2)

    # Random effects
    b = pm.Normal("b", mu=0, sigma=sigma, shape=n_pois)

    # Likelihood
    mu = pm.math.exp(beta0 + b)
    pm.Poisson("y_obs", mu=mu, observed=y_pois)

    # Sample
    trace_pois = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                           progressbar=False)

summary_pois = az.summary(trace_pois, var_names=["beta0", "sigma"])
print(summary_pois)
print(f"\nTrue beta0: {beta0_true_pois}, True sigma: {sigma_true_pois}")

Compare this to the 50+ lines of custom Metropolis-within-Gibbs code in the previous lecture. PyMC specifies the same model in about 10 lines and handles all the MCMC mechanics automatically.

### Centering and Non-Centering

A common issue in hierarchical models is the **funnel geometry**: when $\sigma$ is small, the random effects $b_i$ must also be small, creating a funnel-shaped posterior that is difficult for HMC to navigate. This manifests as **divergent transitions** in the NUTS output.

The fix is **non-centered parameterization**: instead of $b_i \sim N(0, \sigma)$, write $b_i = \sigma \cdot z_i$ where $z_i \sim N(0, 1)$. This decouples the random effects from $\sigma$ in the prior.

In [ ]:
with pm.Model() as poisson_model_nc:
    beta0 = pm.Normal("beta0", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=2)

    # Non-centered parameterization
    z = pm.Normal("z", mu=0, sigma=1, shape=n_pois)
    b = pm.Deterministic("b", sigma * z)

    mu = pm.math.exp(beta0 + b)
    pm.Poisson("y_obs", mu=mu, observed=y_pois)

    trace_nc = pm.sample(2000, tune=1000, chains=4, random_seed=42,
                         progressbar=False)

summary_nc = az.summary(trace_nc, var_names=["beta0", "sigma"])
print("Non-centered parameterization:")
print(summary_nc)

### MCMC Diagnostics with ArviZ

ArviZ provides a comprehensive set of diagnostic plots:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Trace plot for beta0
az.plot_trace(trace_pois, var_names=["beta0"], axes=axes[0:1])

# R-hat across parameters
ax_rhat = axes[1, 0]
rhat_vals = az.rhat(trace_pois, var_names=["beta0", "sigma"])
for var_name in ["beta0", "sigma"]:
    val = float(rhat_vals[var_name].values)
    ax_rhat.barh(var_name, val, color="steelblue")
ax_rhat.axvline(1.01, color="red", linestyle="--", label="$\\hat{R}$ = 1.01")
ax_rhat.set_xlabel("$\\hat{R}$")
ax_rhat.set_title("Gelman-Rubin $\\hat{R}$")
ax_rhat.legend()

# ESS
ax_ess = axes[1, 1]
ess_vals = az.ess(trace_pois, var_names=["beta0", "sigma"])
for var_name in ["beta0", "sigma"]:
    val = float(ess_vals[var_name].values)
    ax_ess.barh(var_name, val, color="steelblue")
ax_ess.axvline(400, color="red", linestyle="--", label="ESS = 400")
ax_ess.set_xlabel("ESS")
ax_ess.set_title("Effective Sample Size")
ax_ess.legend()

plt.tight_layout()

### Question

A researcher builds a hierarchical Bayesian model in PyMC and sees 50 divergent transitions out of 2000 post-warmup samples. The model uses the centered parameterization $b_i \sim N(0, \sigma)$.

(a) What do divergent transitions indicate about the sampler's behavior?

(b) The researcher's colleague suggests increasing `target_accept` from 0.8 to 0.99. Another colleague suggests switching to the non-centered parameterization. Which approach addresses the root cause?

(c) After switching to the non-centered parameterization, the divergences disappear but $\hat{R}$ for $\sigma$ is 1.15. What should the researcher do?

### Answer

(a) Divergent transitions occur when the leapfrog integrator encounters a region of high posterior curvature and the numerical trajectory diverges from the true Hamiltonian trajectory, causing a large energy error. Unlike ordinary MH rejections (which are expected and harmless), divergences are diagnostic warnings that the sampler's numerical integrator has failed in a specific region of the posterior. They indicate that the sampler is systematically unable to explore part of the parameter space, meaning the resulting samples may be biased because regions of the posterior near the divergences are underrepresented. In hierarchical models, this typically happens in the "funnel" region where $\sigma$ is small and the random effects are tightly constrained.

(b) Increasing `target_accept` to 0.99 forces a smaller step size, which may reduce divergences but makes the sampler much slower. It treats the symptom (large step size relative to curvature) rather than the cause (pathological geometry). The non-centered parameterization changes the posterior geometry to remove the funnel, addressing the root cause. It is strongly preferred.

(c) $\hat{R} = 1.15$ indicates the chains have not converged. The researcher should run the chains longer (increase `tune` and `draws`), try different initial values, or check whether the model has identifiability issues. The non-centered parameterization fixed the geometry problem but the chains may need more iterations to fully explore the posterior.

### Practical Workflow Summary

When performing Bayesian analysis with MCMC in practice:

1. **Specify the model** using a probabilistic programming language (PyMC, Stan, NumPyro).

2. **Run the sampler** with default settings (NUTS, 4 chains, 1000 warmup + 1000 draws).

3. **Check diagnostics:**
   - $\hat{R} < 1.01$ for all parameters
   - No divergent transitions
   - ESS $\geq 400$ for bulk and tail (use `az.summary()`)
   - Trace plots look stationary ("fuzzy caterpillar")

4. **If diagnostics fail:**
   - Divergences: reparameterize (non-centered), or increase `target_accept`
   - Low ESS / high $\hat{R}$: run longer, improve parameterization, or simplify the model
   - Slow sampling: consider NumPyro (JAX-based, faster) or reduce model complexity

5. **Report results** including posterior summaries and diagnostics. Always report $\hat{R}$ and ESS.

### Software Comparison

| Feature | PyMC | Stan | NumPyro |
|---------|------|------|---------|
| Language | Python | Stan DSL (compiled to C++) | Python (JAX) |
| Default sampler | NUTS | NUTS | NUTS |
| Discrete parameters | Yes (via Metropolis) | No | Yes (via enumeration) |
| Speed | Moderate | Fast | Fastest (GPU support) |
| Diagnostics | ArviZ | Built-in + ShinyStan | ArviZ |
| Learning curve | Low (Pythonic API) | Moderate (separate language) | Moderate (JAX idioms) |

For this course, we use PyMC because of its Pythonic API and integration with the Python ecosystem. For performance-critical applications, NumPyro (built on JAX) offers significant speed improvements, and Stan has the most mature documentation for modeling best practices.

### NumPyro Example

NumPyro uses a function-based model specification style built on JAX. Here is the same logistic regression model:

In [ ]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

def logistic_model_numpyro(X, y=None):
    beta = numpyro.sample("beta", dist.Normal(jnp.zeros(X.shape[1]), 10.0))
    logits = X @ beta
    numpyro.sample("y_obs", dist.Bernoulli(logits=logits), obs=y)

# Run NUTS
kernel = NUTS(logistic_model_numpyro)
mcmc = MCMC(kernel, num_warmup=1000, num_samples=2000, num_chains=4)
mcmc.run(jax.random.PRNGKey(42), X=jnp.array(X), y=jnp.array(y))

mcmc.print_summary()

The JAX backend compiles the model to efficient machine code and supports GPU acceleration. The API differs from PyMC (explicit random keys, function-based models) but the underlying sampler (NUTS) is the same.

### Question

You are modeling patient survival times with a Bayesian Weibull regression model with 5 fixed effects and 30 hospital-level random intercepts ($d = 37$ parameters including the random effects and variance component).

(a) If you were writing the sampler from scratch, would you choose Gibbs, RWMH, or HMC? Why?

(b) In PyMC, you observe that NUTS takes 3 minutes for 4000 samples (1000 warmup + 1000 draws per chain, 4 chains). Is this reasonable, or should you optimize?

(c) Your collaborator wants to add 500 patient-level random effects to the model ($d = 537$). How would this affect the sampler, and what practical advice would you give?

### Answer

(a) With $d = 37$ and a non-conjugate model (Weibull likelihood is not conjugate to normal priors for the coefficients), most full conditionals do not have standard forms. Gibbs sampling is not an option for most parameters. RWMH with $d = 37$ would require careful tuning of a 37-dimensional proposal, and the optimal step size would be small ($O(37^{-1/2}) \approx 0.16$ relative to the posterior scale). HMC is the best choice: it uses gradients to navigate the 37-dimensional posterior efficiently, and the scaling advantage ($O(d^{5/4})$ vs $O(d^2)$) starts to matter at this dimension.

(b) Three minutes for 4000 total post-warmup samples (1000 per chain) with 37 parameters is reasonable. NUTS typically takes seconds to minutes for models of this size. If diagnostics look good ($\hat{R} < 1.01$, ESS $> 400$, no divergences), there is no need to optimize.

(c) Adding 500 patient-level random effects increases the dimension dramatically. The centered parameterization will likely produce funnel geometries and divergent transitions. Use the non-centered parameterization for the patient-level random effects. Sampling will be slower due to the higher dimension, but NUTS scales much better than alternatives ($O(537^{5/4}) / O(37^{5/4}) \approx 24\times$ slower, compared to $O(537^2)/O(37^2) \approx 211\times$ for RWMH). Consider using NumPyro for speed if PyMC is too slow.

## Summary

1. **Hamiltonian Monte Carlo** uses gradient information and simulated Hamiltonian dynamics to make large, directed proposals. The leapfrog integrator preserves phase-space volume and approximately conserves energy, enabling high acceptance rates even for distant proposals. HMC scales as $O(d^{5/4})$ per independent sample, a major improvement over the $O(d^2)$ scaling of random walk MH. The key tuning parameters are the step size $\varepsilon$ and the number of leapfrog steps $L$.

2. **NUTS** automates the choice of trajectory length by detecting U-turns in the Hamiltonian trajectory. Combined with step-size adaptation (dual averaging) and mass matrix estimation during warmup, NUTS is essentially tuning-free and is the default sampler in modern probabilistic programming tools.

3. **Adaptive MCMC** learns the proposal distribution from the chain's history. The Adaptive Metropolis algorithm of Haario et al. (2001) uses the empirical covariance of past samples to shape the proposal, with the scaling factor $2.38^2/d$ from optimal scaling theory. Diminishing adaptation ensures the chain converges to the correct target despite being non-Markovian. The Robbins-Monro rule provides a simple mechanism for tuning a scalar proposal scale toward a target acceptance rate.

4. **Probabilistic programming tools** (PyMC, Stan, NumPyro) implement NUTS with automatic tuning, making Bayesian inference accessible without writing custom samplers. Key practices include checking diagnostics ($\hat{R}$, ESS, divergences), using non-centered parameterizations for hierarchical models, and running multiple chains from dispersed starting values.

### Connections

- **General MCMC (previous lecture):** HMC is a special case of the Metropolis-Hastings framework with a carefully designed proposal. Adaptive MCMC modifies the MH proposal online while preserving the target distribution.

- **Numerical integration:** MCMC (including HMC) computes posterior expectations by the ergodic theorem, replacing intractable integrals with sample averages. HMC's efficiency advantage means these averages converge faster for a given computational budget.

- **EM algorithm:** Both EM and MCMC handle latent variables. EM finds point estimates (MAP/MLE), while MCMC characterizes the full posterior. For complex hierarchical models, probabilistic programming with NUTS is often simpler than deriving and implementing the E-step and M-step.

- **Optimization:** The gradient $\nabla \log \pi(\boldsymbol{\theta})$ used in HMC is the same gradient used in gradient descent for optimization. HMC repurposes this gradient for sampling rather than optimization. The mass matrix in HMC plays a similar role to preconditioning in optimization.

### References

- Neal, R. M. (2011). MCMC using Hamiltonian dynamics. In *Handbook of Markov Chain Monte Carlo* (eds. S. Brooks, A. Gelman, G. L. Jones, X.-L. Meng), Chapter 5. Chapman and Hall/CRC.
- Hoffman, M. D., & Gelman, A. (2014). The No-U-Turn Sampler: Adaptively setting path lengths in Hamiltonian Monte Carlo. *Journal of Machine Learning Research*, 15, 1593-1623.
- Haario, H., Saksman, E., & Tamminen, J. (2001). An adaptive Metropolis algorithm. *Bernoulli*, 7(2), 223-242.
- Roberts, G. O., & Rosenthal, J. S. (2007). Coupling and ergodicity of adaptive Markov chain Monte Carlo algorithms. *Journal of Applied Probability*, 44(2), 458-475.
- Betancourt, M. (2017). A conceptual introduction to Hamiltonian Monte Carlo. *arXiv:1701.02434*.
- Salvatier, J., Wiecki, T. V., & Fonnesbeck, C. (2016). Probabilistic programming in Python using PyMC3. *PeerJ Computer Science*, 2, e55.